In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
gi_table = pd.read_csv("/kaggle/input/datasets/raesiow/gi-dataset/gi_table_folds.csv", dtype={"code": str})  # load; keep barcodes as text
print(gi_table.shape)                                   # should be (6851, 16)
print(gi_table[["atkinson_name", "gi", "food_weight", "fold"]].head())  # quick look at a few rows

In [ ]:
feature_columns = ["carb_excl_fiber_g", "sugars_g", "fiber_g", "fat_g", "protein_g"]  # the 5 nutrients for the plain MLP
features = gi_table[feature_columns]                    # the inputs (6851 rows x 5 columns)
label = gi_table["gi"]                                  # what we want to predict
weights = gi_table["food_weight"]                       # so each Atkinson food counts once

print(features.describe().round(1).to_string())         # min/max/average of each nutrient, to spot anything strange

In [ ]:
features_log = np.log1p(features)                       # log(1 + x) for every value; 0 stays 0
print(features_log.describe().round(2).to_string())     # values should now be roughly 0 to 4.6

In [ ]:
import torch                                            # PyTorch, the deep learning library
import torch.nn as nn                                   # building blocks for neural networks
from sklearn.preprocessing import StandardScaler        # makes each feature mean 0, std 1

def make_model(n_inputs, h1=32, h2=16, dropout=0.0):        # builds a fresh MLP; defaults = your original model
    return nn.Sequential(                                   # layers run one after another
        nn.Linear(n_inputs, h1), nn.ReLU(), nn.Dropout(dropout),   # inputs -> h1 hidden units, then randomly switch some off
        nn.Linear(h1, h2), nn.ReLU(), nn.Dropout(dropout),         # h1 -> h2 hidden units, same dropout
        nn.Linear(h2, 1))                                   # h2 -> 1 output: the predicted GI
def weighted_mae(true, pred, w):                        # average size of the error, each food counting once
    return np.sum(w * np.abs(true - pred)) / np.sum(w)

def weighted_rmse(true, pred, w):                       # like MAE, but big errors count extra
    return np.sqrt(np.sum(w * (true - pred) ** 2) / np.sum(w))

In [ ]:
def train_one_fold(x_train, y_train, w_train, x_test, h1=32, h2=16, dropout=0.0):  # trains a fresh model, returns test predictions
    torch.manual_seed(42)                                   # same starting weights every time
    model = make_model(x_train.shape[1], h1, h2, dropout)   # fresh model with the chosen size and dropout
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # how the weights get updated
    x = torch.tensor(x_train, dtype=torch.float32)          # training inputs as a PyTorch tensor
    y = torch.tensor(y_train, dtype=torch.float32)          # training labels
    w = torch.tensor(w_train, dtype=torch.float32)          # training weights
    model.train()                                           # NEW: training mode, so dropout is switched ON
    for epoch in range(1000):                               # 1000 passes over all training rows
        optimizer.zero_grad()                               # clear old gradients
        pred = model(x).squeeze(1)                          # predict for all rows
        loss = (w * (pred - y) ** 2).sum() / w.sum()        # weighted squared error
        loss.backward()                                     # work out how to change each weight
        optimizer.step()                                    # change the weights a little
    model.eval()                                            # NEW: prediction mode, so dropout is switched OFF
    with torch.no_grad():                                   # no learning from here on
        return model(torch.tensor(x_test, dtype=torch.float32)).squeeze(1).numpy()  # test predictions

In [ ]:
results = []                                            # one row of scores per fold
gi_table["pred_plain"] = np.nan                         # column to store each product's out-of-fold prediction

for fold in range(5):                                   # each fold takes a turn as the test set
    train = gi_table["fold"] != fold                    # True for training rows
    test = gi_table["fold"] == fold                     # True for test rows

    scaler = StandardScaler().fit(features_log[train])  # learn mean/std from TRAINING rows only
    x_train = scaler.transform(features_log[train])     # scale training features
    x_test = scaler.transform(features_log[test])       # scale test features with the same numbers

    gi_mean = np.average(label[train], weights=weights[train])  # weighted average GI in training
    gi_std = label[train].std()                         # spread of GI in training
    y_train = ((label[train] - gi_mean) / gi_std).values  # scaled labels (around 0), easier to learn

    pred = train_one_fold(x_train, y_train, weights[train].values, x_test) * gi_std + gi_mean  # predict, then undo the scaling
    gi_table.loc[test, "pred_plain"] = pred             # store predictions

    true, w = label[test].values, weights[test].values  # true GI and weights of the test rows
    results.append({"fold": fold,
                    "mae": weighted_mae(true, pred, w),
                    "rmse": weighted_rmse(true, pred, w),
                    "baseline_mae": weighted_mae(true, gi_mean, w)})  # baseline: always guess the average GI
    print("fold", fold,                                         # which fold
          "MAE", round(float(results[-1]["mae"]), 1),           # float() makes it a plain number, round to 1 decimal
          "RMSE", round(float(results[-1]["rmse"]), 1),
          "baseline MAE", round(float(results[-1]["baseline_mae"]), 1))                                 # show this fold's scores

results = pd.DataFrame(results)                         # turn into a table
print(results.mean().round(1))                          # average over the 5 folds
print(results.std().round(1))                           # spread over the 5 folds

In [ ]:
print(results.drop(columns="fold").mean().round(1))    # average scores over the 5 folds (fold column left out)
print(results.drop(columns="fold").std().round(1))     # spread of scores over the 5 folds

In [ ]:
nova_probs = pd.read_csv("/kaggle/input/datasets/raesiow/gi-nova-probs/gi_nova_probs.csv",  # read the NOVA probabilities file
                         dtype={"code": str})                      # keep barcodes as text so leading zeros survive

print(len(gi_table), len(nova_probs))                              # both should print 6851
print((gi_table["code"].astype(str).values ==                      # compare barcodes row by row...
       nova_probs["code"].values).all())                           # ...True means every row lines up

for col in ["p1", "p2", "p3", "p4"]:                               # go through the four NOVA probability columns
    gi_table[col] = nova_probs[col].values                         # copy each one into gi_table by row position

prob_sums = gi_table[["p1", "p2", "p3", "p4"]].sum(axis=1)         # add up the 4 probs for each product
print(prob_sums.min(), prob_sums.max())                            # both should be very close to 1
print(gi_table[["p1", "p2", "p3", "p4"]].isna().sum().sum())       # number of missing probs, should be 0

In [ ]:
nova_cols = ["p1", "p2", "p3", "p4"]                               # CHANGED: the 4 NOVA probability columns
features_fused = pd.concat([features_log,                          # CHANGED: the 5 log1p nutrients (same as plain)...
                            gi_table[nova_cols]], axis=1)          # ...plus the 4 probs side by side = 9 columns
print(features_fused.shape)                                        # should print (6851, 9)

results_fused = []                                                 # CHANGED: new list, so the plain results stay safe
gi_table["pred_fused"] = np.nan                                    # empty column to store each product's out-of-fold fused prediction

for fold in range(5):                                              # same 5 folds as the plain model
    train = gi_table["fold"] != fold                               # True for training rows
    test = gi_table["fold"] == fold                                # True for test rows

    scaler = StandardScaler().fit(features_fused[train])           # learn mean/std from TRAINING rows only
    x_train = scaler.transform(features_fused[train])              # scale training features
    x_test = scaler.transform(features_fused[test])                # scale test features with the same numbers

    gi_mean = np.average(label[train], weights=weights[train])     # weighted average GI in training
    gi_std = label[train].std()                                    # spread of GI in training
    y_train = ((label[train] - gi_mean) / gi_std).values           # scaled labels (around 0)

    pred = train_one_fold(x_train, y_train, weights[train].values, x_test) * gi_std + gi_mean  # predict, undo scaling
    gi_table.loc[test, "pred_fused"] = pred                        # store fused predictions in their own column

    true, w = label[test].values, weights[test].values             # true GI and weights of the test rows
    results_fused.append({"fold": fold,                            # this fold's number
                          "mae": weighted_mae(true, pred, w),      # fused model's weighted MAE
                          "rmse": weighted_rmse(true, pred, w)})   # fused model's weighted RMSE
    print("fold", fold,                                            # which fold
          "MAE", round(float(results_fused[-1]["mae"]), 1),        # this fold's MAE, 1 decimal
          "RMSE", round(float(results_fused[-1]["rmse"]), 1))      # this fold's RMSE, 1 decimal

results_fused = pd.DataFrame(results_fused)                        # turn the list into a table
print(results_fused.drop(columns="fold").mean().round(1))          # average over the 5 folds
print(results_fused.drop(columns="fold").std().round(1))           # spread over the 5 folds

In [ ]:
gi_table.to_csv("/kaggle/working/gi_fused_preds.csv", index=False)               # all products with pred_plain AND pred_fused
results_fused.to_csv("/kaggle/working/fused_mlp_results.csv", index=False)       # fused per-fold scores for the report

compare = pd.DataFrame({"fold": results["fold"],                                 # one row per fold
                        "baseline_mae": results["baseline_mae"],                 # always-guess-average scores
                        "plain_mae": results["mae"],                             # plain MLP MAE
                        "fused_mae": results_fused["mae"],                       # fused MLP MAE
                        "plain_rmse": results["rmse"],                           # plain MLP RMSE
                        "fused_rmse": results_fused["rmse"]})                    # fused MLP RMSE
compare["fused_better_mae"] = compare["fused_mae"] < compare["plain_mae"]        # True where fused wins on MAE
print(compare.round(1))                                                          # the paired table
print("fused wins on MAE in", compare["fused_better_mae"].sum(), "of 5 folds")   # count of wins
compare.to_csv("/kaggle/working/ablation_table.csv", index=False)                # save the ablation table

In [ ]:
def train_error(features):                                           # weighted training MAE, one number per fold
    errors = []                                                      # empty list for this model's 5 scores
    for fold in range(5):                                            # same 5 folds as before
        train = gi_table["fold"] != fold                             # True for training rows
        scaler = StandardScaler().fit(features[train])               # same scaling as the real run
        x_train = scaler.transform(features[train])                  # scaled training features
        gi_mean = np.average(label[train], weights=weights[train])   # same label scaling as before
        gi_std = label[train].std()                                  # same spread as before
        y_train = ((label[train] - gi_mean) / gi_std).values         # scaled training labels
        pred = train_one_fold(x_train, y_train, weights[train].values, x_train) * gi_std + gi_mean  # predict the TRAINING rows
        errors.append(weighted_mae(label[train].values, pred, weights[train].values))  # how well it fits foods it has seen
    return errors                                                    # list of 5 training MAEs

compare["plain_train_mae"] = train_error(features_log)               # plain model: 5 nutrients
compare["fused_train_mae"] = train_error(features_fused)             # fused model: 5 nutrients + 4 probs
print(compare[["fold", "plain_train_mae", "plain_mae",               # training vs test error, side by side
               "fused_train_mae", "fused_mae"]].round(1))
compare.to_csv("/kaggle/working/ablation_table.csv", index=False)    # re-save the table with the new columns

In [ ]:
def run_cv(features):                                                # runs the 5-fold test for any set of input columns
    maes, rmses = [], []                                             # empty lists for this variant's scores
    for fold in range(5):                                            # same 5 folds as always
        train = gi_table["fold"] != fold                             # True for training rows
        test = gi_table["fold"] == fold                              # True for test rows
        scaler = StandardScaler().fit(features[train])               # scaling learned from training rows only
        x_train = scaler.transform(features[train])                  # scaled training features
        x_test = scaler.transform(features[test])                    # scaled test features
        gi_mean = np.average(label[train], weights=weights[train])   # weighted average GI in training
        gi_std = label[train].std()                                  # spread of GI in training
        y_train = ((label[train] - gi_mean) / gi_std).values         # scaled training labels
        pred = train_one_fold(x_train, y_train, weights[train].values, x_test) * gi_std + gi_mean  # train + predict test rows
        maes.append(weighted_mae(label[test].values, pred, weights[test].values))    # this fold's MAE
        rmses.append(weighted_rmse(label[test].values, pred, weights[test].values))  # this fold's RMSE
    return maes, rmses                                               # 5 MAEs and 5 RMSEs

log_probs = np.log(gi_table[nova_cols].clip(lower=1e-6))             # log of probs = logits up to a per-row shift; clip avoids log(0)
extra_nut = np.log1p(gi_table[["sat_fat_g", "sodium_g"]])            # 2 extra nutrients, log1p like the other 5

variants = {"fused_logits": pd.concat([features_log, log_probs], axis=1),                       # 5 nutrients + 4 log-probs
            "plain_plus2": pd.concat([features_log, extra_nut], axis=1),                        # 7 nutrients
            "fused_plus2": pd.concat([features_log, extra_nut, gi_table[nova_cols]], axis=1)}   # 7 nutrients + 4 probs

for name, feats in variants.items():                                 # run each variant in turn
    print(name, feats.shape)                                         # check the number of columns
    compare[name + "_mae"], compare[name + "_rmse"] = run_cv(feats)  # run 5 folds, add scores as new columns

print(compare.filter(like="mae").drop(columns=["baseline_mae"]).mean().round(1))   # average MAE of every model
compare.to_csv("/kaggle/working/ablation_table.csv", index=False)                  # save everything (re-download after)

In [ ]:
settings = {"A": (32, 16, 0.0),                               # current model: 32 -> 16, no dropout
            "B": (32, 16, 0.3),                               # same size, 30% dropout
            "C": (64, 32, 0.0),                               # bigger, no dropout
            "D": (64, 32, 0.3)}                               # bigger, 30% dropout

def fit_predict(features, train, test, setting):              # scale, train one model, predict the test rows
    h1, h2, dropout = setting                                 # unpack the 3 numbers of this setting
    scaler = StandardScaler().fit(features[train])            # scaling learned from training rows only
    x_train = scaler.transform(features[train])               # scaled training features
    x_test = scaler.transform(features[test])                 # scaled test features
    gi_mean = np.average(label[train], weights=weights[train])  # weighted average GI in training
    gi_std = label[train].std()                               # spread of GI in training
    y_train = ((label[train] - gi_mean) / gi_std).values      # scaled training labels
    pred = train_one_fold(x_train, y_train, weights[train].values, x_test, h1, h2, dropout)  # train with this setting
    return pred * gi_std + gi_mean                            # undo the label scaling

def nested_cv(features, model_name):                          # honest score with settings chosen inside each fold
    rows = []                                                 # one row of results per outer fold
    for outer in range(5):                                    # each fold takes a turn as the untouched test set
        outer_train = gi_table["fold"] != outer               # the other 4 folds
        outer_test = gi_table["fold"] == outer                # the test fold (not looked at while choosing)
        inner_scores = {}                                     # average inner MAE for each setting
        for name, setting in settings.items():                # try each of the 4 settings
            fold_maes = []                                    # inner MAEs for this setting
            for inner in range(5):                            # loop over fold numbers...
                if inner == outer:                            # ...skipping the outer test fold
                    continue                                  # go to the next fold number
                inner_train = outer_train & (gi_table["fold"] != inner)  # 3 folds to train on
                inner_check = gi_table["fold"] == inner       # 1 fold to check this setting on
                pred = fit_predict(features, inner_train, inner_check, setting)  # train + predict the check fold
                fold_maes.append(weighted_mae(label[inner_check].values, pred, weights[inner_check].values))  # its MAE
            inner_scores[name] = np.mean(fold_maes)           # average over the 4 inner checks
        best = min(inner_scores, key=inner_scores.get)        # setting with the lowest inner MAE
        pred = fit_predict(features, outer_train, outer_test, settings[best])  # retrain best setting on all 4 folds
        true, w = label[outer_test].values, weights[outer_test].values  # true GI and weights of the test fold
        rows.append({"fold": outer, "chosen": best,           # which setting this fold picked
                     "mae": weighted_mae(true, pred, w),      # honest test MAE
                     "rmse": weighted_rmse(true, pred, w)})   # honest test RMSE
        print(model_name, "fold", outer, "chose", best,       # progress line for each fold
              {k: round(float(v), 1) for k, v in inner_scores.items()},  # inner scores of all 4 settings
              "test MAE", round(float(rows[-1]["mae"]), 1))   # this fold's honest test MAE
    return pd.DataFrame(rows)                                 # table of the 5 outer folds

tuned_plain = nested_cv(features_log, "plain")                # tune + test the plain model (5 nutrients)
tuned_fused = nested_cv(features_fused, "fused")              # tune + test the fused model (5 nutrients + 4 probs)

for name, table in [("plain", tuned_plain), ("fused", tuned_fused)]:  # summary for both models
    print(name, "tuned MAE", round(table["mae"].mean(), 1), "+/-", round(table["mae"].std(), 1),     # mean +/- spread MAE
          "RMSE", round(table["rmse"].mean(), 1), "+/-", round(table["rmse"].std(), 1))              # mean +/- spread RMSE

tuned_plain.to_csv("/kaggle/working/tuned_plain_results.csv", index=False)  # save plain tuned results
tuned_fused.to_csv("/kaggle/working/tuned_fused_results.csv", index=False)  # save fused tuned results

In [ ]:
gi_table["error_plain"] = gi_table["pred_plain"] - gi_table["gi"]   # positive = predicted too high, negative = too low
gi_table["abs_error_plain"] = gi_table["error_plain"].abs()         # size of the error, ignoring direction

per_food = gi_table.groupby("merge_key").agg(                       # one row per Atkinson food
    atkinson_name=("atkinson_name", "first"),                       # the food's name
    label_gi=("gi", "first"),                                       # its GI label (same for all its products)
    mean_pred=("pred_plain", "mean"),                               # average prediction over its products
    mean_abs_error=("abs_error_plain", "mean"),                     # average error size
    n_products=("code", "count"))                                   # how many OFF products it has

pd.set_option("display.max_colwidth", 40)                           # cut long names so the table fits
print(per_food.sort_values("mean_abs_error", ascending=False).head(15).round(1))   # 15 worst foods

In [ ]:
food = "hummus"                                                 # paste a merge_key from the table above
one_food = gi_table[gi_table["merge_key"] == food]                  # all products matched to that food
print(one_food[["product_name", "gi", "pred_plain", "pred_fused",   # name, label, both predictions...
                "carb_excl_fiber_g", "sugars_g", "fiber_g"]]        # ...and a few nutrients to see why
      .head(20).round(1))                                           # first 20 products

In [ ]:
def gi_band(gi):                                                  # turns GI numbers into the standard bands
    return pd.cut(gi, bins=[-np.inf, 55, 69, np.inf],             # <=55 low, 56-69 medium, >=70 high
                  labels=["low", "medium", "high"])               # names for the 3 bands

true_band = gi_band(gi_table["gi"])                               # each product's true band
w = gi_table["food_weight"]                                       # each food counts once, as in MAE

print(true_band.value_counts())                                   # how many products fall in each band
for col in ["pred_plain", "pred_fused"]:                          # check both models
    pred_band = gi_band(gi_table[col])                            # predicted band
    acc = np.average(pred_band == true_band, weights=w)           # weighted share of right bands
    print(col, "band accuracy", round(acc, 3))                    # e.g. 0.6 = 60% right

base_acc = np.average(true_band == "low", weights=w)              # baseline: mean GI ~53 is always "low"
print("baseline (always low)", round(base_acc, 3))                # what guessing the average gets

print(pd.crosstab(true_band, gi_band(gi_table["pred_plain"]),     # rows = true band, columns = predicted band
                  values=w, aggfunc="sum").round(1))              # weighted counts (number of foods)

In [ ]:
for col in ["pred_plain", "pred_fused"]:                                  # check both models
    mse = np.average((gi_table["gi"] - gi_table[col]) ** 2, weights=w)    # weighted squared error
    var = np.average((gi_table["gi"] - np.average(gi_table["gi"], weights=w)) ** 2, weights=w)  # weighted spread of GI
    print(col, "R2", round(1 - mse / var, 3))                             # share of GI variation explained

In [ ]:
print(gi_table.shape)                                          # should be (6851, ...) if the session is still alive
print("pred_plain" in gi_table, "pred_fused" in gi_table)      # True True = both predictions still there

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor              # gradient-boosted trees for numbers (regression)

gbm_rows = []                                                           # one row of scores per fold
gi_table["pred_gbm"] = np.nan                                           # column for each product's out-of-fold prediction
for fold in range(5):                                                   # same 5 grouped folds as the MLPs
    train = gi_table["fold"] != fold                                    # True for training rows
    test = gi_table["fold"] == fold                                     # True for test rows
    gbm = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05,   # settings fixed in advance (same as NOVA GBM)
                                        random_state=42)                # same seed every run
    gbm.fit(features_log[train], label[train],                          # learn from training foods only...
            sample_weight=weights[train])                               # ...each food counting once
    pred = gbm.predict(features_log[test])                              # predict the unseen test foods
    gi_table.loc[test, "pred_gbm"] = pred                               # store predictions
    true, w_test = label[test].values, weights[test].values             # true GI and weights of the test rows
    gbm_rows.append({"fold": fold,                                      # which fold
                     "mae": weighted_mae(true, pred, w_test),           # weighted MAE
                     "rmse": weighted_rmse(true, pred, w_test)})        # weighted RMSE

gbm_results = pd.DataFrame(gbm_rows)                                    # turn into a table
print(gbm_results.round(1))                                             # per-fold scores
print("GBM MAE", round(gbm_results["mae"].mean(), 1), "+/-", round(gbm_results["mae"].std(), 1))  # average +/- spread

mse = np.average((gi_table["gi"] - gi_table["pred_gbm"]) ** 2, weights=w)   # weighted squared error
var = np.average((gi_table["gi"] - np.average(gi_table["gi"], weights=w)) ** 2, weights=w)  # weighted spread of GI
print("GBM R2", round(1 - mse / var, 3))                                # compare: plain 0.123
gbm_results.to_csv("/kaggle/working/gbm_gi_results.csv", index=False)  # save for the report

In [ ]:
from sentence_transformers import SentenceTransformer             # library for text embeddings (used it in the MiniLM notebook)

minilm = SentenceTransformer("all-MiniLM-L6-v2")                  # small English text model: each text -> 384 numbers
texts = gi_table["ingredients_text"].fillna("").tolist()          # the 6,851 ingredient texts as a plain list
embeddings = minilm.encode(texts, batch_size=64,                  # encode 64 texts at a time...
                           show_progress_bar=True)                # ...with a progress bar so you can see it working
print(embeddings.shape)                                           # should be (6851, 384)
np.save("/kaggle/working/gi_text_embeddings.npy", embeddings)     # save so you never need to encode again

In [ ]:
from sklearn.decomposition import PCA                                    # shrinks 384 text numbers down to a few

extra_nut = np.log1p(gi_table[["sat_fat_g", "sodium_g"]])                # 2 extra nutrients, log1p like the other 5
features_7 = pd.concat([features_log, extra_nut], axis=1)                # 7 nutrients for the MLP runs

def cv_text(nut_features, use_text, model_type):                        # 5-fold test: nutrients, optionally + text, GBM or MLP
    rows = []                                                            # one row of scores per fold
    preds = np.full(len(gi_table), np.nan)                               # out-of-fold prediction for every product
    for fold in range(5):                                                # same 5 grouped folds as always
        train = (gi_table["fold"] != fold).values                        # True for training rows (as a numpy array)
        test = (gi_table["fold"] == fold).values                         # True for test rows
        x_train = nut_features.values[train]                             # training nutrients
        x_test = nut_features.values[test]                               # test nutrients
        if use_text:                                                     # only for the "+ text" runs
            pca = PCA(n_components=16, random_state=42).fit(embeddings[train])  # learn the 16 main text directions from TRAINING rows only
            x_train = np.hstack([x_train, pca.transform(embeddings[train])])    # join 16 text numbers to the training nutrients
            x_test = np.hstack([x_test, pca.transform(embeddings[test])])       # same for test rows, using the training PCA
        if model_type == "gbm":                                          # gradient-boosted trees
            gbm = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, random_state=42)  # same fixed settings as before
            gbm.fit(x_train, label.values[train], sample_weight=weights.values[train])  # learn, each food counting once
            pred = gbm.predict(x_test)                                   # predict unseen test foods
        else:                                                            # the MLP
            scaler = StandardScaler().fit(x_train)                       # scaling learned from training rows only
            x_train_s, x_test_s = scaler.transform(x_train), scaler.transform(x_test)  # scaled inputs
            gi_mean = np.average(label.values[train], weights=weights.values[train])   # weighted average GI in training
            gi_std = label.values[train].std()                           # spread of GI in training
            y_train = (label.values[train] - gi_mean) / gi_std           # scaled training labels
            pred = train_one_fold(x_train_s, y_train, weights.values[train], x_test_s,
                                  64, 32, 0.5) * gi_std + gi_mean        # setting D, fixed in advance; undo label scaling
        preds[test] = pred                                               # store this fold's predictions
        true, w_test = label.values[test], weights.values[test]          # true GI and weights of the test rows
        rows.append({"fold": fold, "mae": weighted_mae(true, pred, w_test),    # weighted MAE
                     "rmse": weighted_rmse(true, pred, w_test)})               # weighted RMSE
    return pd.DataFrame(rows), preds                                     # per-fold scores + all predictions

runs = [("gbm_5nut", features_log, False, "gbm"),                        # GBM, nutrients only (should repeat 11.8)
        ("gbm_5nut_text", features_log, True, "gbm"),                    # GBM, nutrients + text
        ("mlp_7nut", features_7, False, "mlp"),                          # MLP setting D, 7 nutrients
        ("mlp_7nut_text", features_7, True, "mlp")]                      # MLP setting D, 7 nutrients + text

var = np.average((label - np.average(label, weights=w)) ** 2, weights=w) # weighted spread of GI (for R2)
summary = []                                                             # one summary row per run
for name, feats, use_text, model_type in runs:                           # go through the 4 runs
    scores, preds = cv_text(feats, use_text, model_type)                 # run 5 folds
    gi_table["pred_" + name] = preds                                     # keep predictions for later (bands, plots)
    r2 = 1 - np.average((label - preds) ** 2, weights=w) / var           # share of GI variation explained
    summary.append({"run": name, "mae": scores["mae"].mean(), "mae_std": scores["mae"].std(),  # average and spread
                    "rmse": scores["rmse"].mean(), "r2": r2,             # RMSE and R2
                    "fold_maes": list(scores["mae"].round(1))})          # the 5 fold MAEs, for the paired comparison
    print(name, "MAE", round(scores["mae"].mean(), 1), list(scores["mae"].round(1)))  # progress line

summary = pd.DataFrame(summary)                                          # turn into a table
print(summary.round(3).to_string())                                      # full comparison
summary.to_csv("/kaggle/working/text_embedding_results.csv", index=False)  # save for the report

In [ ]:
merged = pd.read_csv('/kaggle/input/datasets/raesiow/gi-dataset/atkinson_merged.csv')
matched = pd.read_csv('/kaggle/input/datasets/raesiow/gi-dataset/atkinson_off_fuzzy_matches.csv')
print(merged.shape, matched.shape)                    # number of rows and columns of each
print(merged.columns.tolist())                        # column names of the merged table
print(matched.columns.tolist())                       # column names of the matches table
print(merged.head(3).to_string())     

In [ ]:
info = merged.set_index("merge_key")                                          # look up Atkinson info by food key
gi_table["atk_carb"] = gi_table["merge_key"].map(info["avail_carb_per_100g"])  # Atkinson carbs for each product's food
gi_table["gi_range"] = gi_table["merge_key"].map(info["gi_max"] - info["gi_min"])  # GI spread of each product's merged food
print("missing ranges:", gi_table["gi_range"].isna().sum())                   # should be 0 (every food found)

ratio = gi_table["carb_excl_fiber_g"] / gi_table["atk_carb"]                  # product carbs / Atkinson carbs
form_mismatch = ratio.notna() & ((ratio < 0.5) | (ratio > 2))                 # rule 1: less than half or more than double
noisy_label = gi_table["gi_range"] > 30                                       # rule 2: merged food's GI range over 30 points
print("form mismatch products:", form_mismatch.sum(), "| noisy label products:", noisy_label.sum())  # how many each rule flags

def gbm_clean_cv(keep):                                                       # GBM where only kept rows are used for TRAINING
    maes = []                                                                 # one MAE per fold
    for fold in range(5):                                                     # same 5 grouped folds
        train = ((gi_table["fold"] != fold) & keep).values                    # training rows that pass the cleaning
        test = (gi_table["fold"] == fold).values                              # ALL test rows, same as always
        gbm = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, random_state=42)  # same fixed settings
        gbm.fit(features_log.values[train], label.values[train], sample_weight=weights.values[train])  # learn from clean rows
        pred = gbm.predict(features_log.values[test])                         # predict every test row
        maes.append(weighted_mae(label.values[test], pred, weights.values[test]))  # weighted MAE on the unchanged test fold
    return maes                                                               # 5 MAEs

all_rows = pd.Series(True, index=gi_table.index)                              # "keep everything" (no cleaning)
for name, keep in [("no cleaning", all_rows),                                 # should repeat 11.8
                   ("drop form mismatch", ~form_mismatch),                    # rule 1 only
                   ("drop noisy labels", ~noisy_label),                       # rule 2 only
                   ("drop both", ~form_mismatch & ~noisy_label)]:             # both rules
    maes = gbm_clean_cv(keep)                                                 # run 5 folds
    print(name, "MAE", round(np.mean(maes), 2), [round(m, 1) for m in maes])  # average + per-fold MAEs

In [ ]:
atk_names = merged["example_item"].fillna(merged["name_head"]).tolist()      # full Atkinson descriptions (name_head if missing)
atk_emb = minilm.encode(atk_names, batch_size=64, normalize_embeddings=True)    # 2,590 Atkinson foods -> 384 numbers each
atk_gi = merged["gi_median"].values                                           # clean GI label of each Atkinson food
atk_keys = merged["merge_key"].values                                         # food ID of each Atkinson food

name_only = gi_table["product_name"].fillna("")                               # version 1: OFF product name
name_ingr = name_only + ". " + gi_table["ingredients_text"].fillna("")        # version 2: name + ingredient text
off_versions = {"name": minilm.encode(name_only.tolist(), batch_size=64, normalize_embeddings=True),              # encode version 1
                "name+ingredients": minilm.encode(name_ingr.tolist(), batch_size=64, normalize_embeddings=True)}  # encode version 2

K = 5                                                                         # number of neighbours, fixed in advance

def knn_cv(off_emb):                                                          # name-based kNN, 5 grouped folds
    maes = []                                                                 # one MAE per fold
    preds = np.full(len(gi_table), np.nan)                                    # prediction for every product
    for fold in range(5):                                                     # same 5 folds as always
        test = (gi_table["fold"] == fold).values                              # test products of this fold
        test_foods = set(gi_table.loc[test, "merge_key"])                     # this fold's Atkinson foods (hidden)
        pool = ~np.isin(atk_keys, list(test_foods))                           # all other Atkinson foods (~2,460)
        sims = off_emb[test] @ atk_emb[pool].T                                # similarity of each test product to each pool food
        top = np.argsort(-sims, axis=1)[:, :K]                                # 5 most similar foods per product
        top_sims = np.clip(np.take_along_axis(sims, top, axis=1), 0, None)    # their similarities (negatives set to 0)
        top_gi = atk_gi[pool][top]                                            # their GI labels
        pred = (top_sims * top_gi).sum(axis=1) / (top_sims.sum(axis=1) + 1e-9)  # similarity-weighted average GI
        preds[test] = pred                                                    # store this fold's predictions
        maes.append(weighted_mae(label.values[test], pred, weights.values[test]))  # weighted MAE on this fold
    return maes, preds                                                        # 5 MAEs + all predictions

for name, emb in off_versions.items():                                        # run both versions
    maes, preds = knn_cv(emb)                                                 # 5-fold kNN
    gi_table["pred_knn_" + name] = preds                                      # keep predictions for later
    print(name, "kNN MAE", round(np.mean(maes), 2), "+/-",                    # average...
          round(np.std(maes, ddof=1), 2), [round(m, 1) for m in maes])        # ...spread and per-fold MAEs

In [ ]:
gi_table.to_csv("/kaggle/working/gi_all_preds.csv", index=False)   # save every product with all of tonight's prediction columns
print(gi_table.shape)                                                # quick check: 6,851 rows, plus the column count

In [ ]:
needed = ["np", "gi_table", "label", "weights", "weighted_mae", "merged", "minilm"]  # everything the kNN cell needs
for name in needed:                                                    # check them one by one
    print(name, "OK" if name in globals() else "MISSING")              # OK = in memory, MISSING = find its cell

In [ ]:
rng = np.random.default_rng(42)                               # fixed seed so we always see the same 15
foods = gi_table["merge_key"].unique()                        # the 629 Atkinson foods in the GI table
sample_foods = rng.choice(foods, size=15, replace=False)      # 15 random foods (the MAE counts each food once)
sample_rows = [np.where(gi_table["merge_key"].values == f)[0][0] for f in sample_foods]  # first product of each food

key_to_name = dict(zip(atk_keys, atk_names))                  # merge_key -> full Atkinson description
names_arr = np.array(atk_names)                               # names as an array so we can index with a mask
name_emb = off_versions["name"]                               # product-name embeddings (the 11.5 version)

for row in sample_rows:                                       # one sampled product at a time
    fold = gi_table["fold"].iloc[row]                         # the fold this product is tested in
    test_foods = set(gi_table.loc[gi_table["fold"] == fold, "merge_key"])  # foods hidden in that fold
    pool = ~np.isin(atk_keys, list(test_foods))               # same pool knn_cv used for this fold
    sims = atk_emb[pool] @ name_emb[row]                      # similarity of this product to every pool food
    top = np.argsort(-sims)[:K]                               # positions of the 5 most similar pool foods
    own_key = gi_table["merge_key"].iloc[row]                 # the product's own (hidden) Atkinson food
    print("=" * 80)                                           # separator line between products
    print("PRODUCT:", gi_table["product_name"].iloc[row], "| fold", fold)       # what the shopper scanned
    print("  own food:", key_to_name[own_key],                # its true Atkinson food...
          "| true GI", gi_table["gi"].iloc[row],              # ...its true GI...
          "| kNN guess", round(gi_table["pred_knn_name"].iloc[row], 1))  # ...and what kNN predicted
    for i in top:                                             # the 5 neighbours kNN averaged
        print(f"    sim {sims[i]:.2f} | GI {atk_gi[pool][i]:5.0f} | {names_arr[pool][i]}")  # similarity, GI, name

In [ ]:
mpnet = SentenceTransformer("all-mpnet-base-v2")                    # bigger text model (768 numbers per name instead of 384)
atk_emb = mpnet.encode(atk_names, batch_size=64,                    # re-encode the 2,590 Atkinson names...
                       normalize_embeddings=True)                   # ...scaled so similarity = dot product, as before
mpnet_name_emb = mpnet.encode(name_only.tolist(), batch_size=64,    # encode the 6,851 product names...
                              normalize_embeddings=True)            # ...same scaling
maes, preds = knn_cv(mpnet_name_emb)                                # same kNN, same folds, same K = 5
gi_table["pred_knn_mpnet"] = preds                                  # keep mpnet predictions in a NEW column (MiniLM's stay safe)
print("mpnet name kNN MAE", round(np.mean(maes), 2), "+/-",         # average over 5 folds...
      round(np.std(maes, ddof=1), 2), [round(float(m), 1) for m in maes])  # ...spread and per-fold MAEs

In [ ]:
knn_cols = ["code", "merge_key", "gi", "food_weight", "fold",       # columns that identify each product...
            "pred_knn_name", "pred_knn_name+ingredients",           # ...MiniLM kNN predictions (both versions)...
            "pred_knn_mpnet"]                                       # ...and the mpnet kNN predictions
gi_table[knn_cols].to_csv("/kaggle/working/gi_knn_preds.csv",      # save just these columns...
                          index=False)                              # ...without the row-number column
print(gi_table[knn_cols].shape)                                     # should be (6851, 8)
print(gi_table[knn_cols].isna().sum().sum())                        # missing values, should be 0

In [ ]:
# list the names we need for tonight's experiment
needed = ["gi_table", "features_log", "make_model", "train_one_fold", "weighted_mae", "weighted_rmse"]
# go through each name one by one
for name in needed:
    # True = still in memory, False = session reset, cell needs rerunning
    print(name, name in globals())

In [ ]:
# KFold splits rows randomly and ignores which food they belong to
from sklearn.model_selection import KFold
# 5 folds, shuffled, fixed seed so the split is the same every run
kf_random = KFold(n_splits=5, shuffle=True, random_state=42)
# new column for the random fold number; -1 = not assigned yet
gi_table["fold_random"] = -1
# k = fold number, test_idx = row positions that go to test in fold k
for k, (train_idx, test_idx) in enumerate(kf_random.split(gi_table)):
    # write fold number k into the test rows of this fold
    gi_table.loc[gi_table.index[test_idx], "fold_random"] = k

# check: products per random fold (expect about 1,370 each, no -1 left)
print(gi_table["fold_random"].value_counts().sort_index())

# check: for each fold, how many test foods also appear in training
for k in range(5):
    # foods that have at least one product in this test fold
    test_foods = set(gi_table.loc[gi_table["fold_random"] == k, "merge_key"])
    # foods that have at least one product in the training folds
    train_foods = set(gi_table.loc[gi_table["fold_random"] != k, "merge_key"])
    # print: fold, number of test foods, how many of those are also in training
    print(k, len(test_foods), len(test_foods & train_foods))

In [ ]:
# the same 5 nutrient inputs as the plain MLP
feature_cols = ["carb_excl_fiber_g", "sugars_g", "fiber_g", "fat_g", "protein_g"]

# runs the plain MLP on every fold of the fold column we give it
def run_split(fold_col):
    # one result row per fold goes in here
    rows = []
    # go through each fold number in that column
    for k in sorted(gi_table[fold_col].unique()):
        # True for rows that are in the test fold k
        is_test = gi_table[fold_col] == k
        # all other rows are training rows
        train = gi_table[~is_test]
        # test rows of fold k
        test = gi_table[is_test]
        # log1p of training nutrients (same as before)
        x_train = np.log1p(train[feature_cols].values)
        # log1p of test nutrients
        x_test = np.log1p(test[feature_cols].values)
        # learn mean/std from TRAINING rows only
        scaler = StandardScaler().fit(x_train)
        # scale training inputs
        x_train = scaler.transform(x_train)
        # scale test inputs with the training mean/std
        x_test = scaler.transform(x_test)
        # training GI labels
        y_train = train["gi"].values
        # training food weights
        w_train = train["food_weight"].values
        # weighted mean GI of the training rows
        mean = np.average(y_train, weights=w_train)
        # weighted std of training GI
        std = np.sqrt(np.average((y_train - mean) ** 2, weights=w_train))
        # train on scaled labels, then unscale the predictions
        pred = train_one_fold(x_train, (y_train - mean) / std, w_train, x_test) * std + mean
        # true GI of the test rows
        y_test = test["gi"].values
        # test food weights
        w_test = test["food_weight"].values
        # weighted MAE
        mae = np.average(np.abs(pred - y_test), weights=w_test)
        # weighted RMSE
        rmse = np.sqrt(np.average((pred - y_test) ** 2, weights=w_test))
        # baseline: always predict the training mean
        base = np.average(np.abs(mean - y_test), weights=w_test)
        # save this fold's scores
        rows.append({"fold": k, "mae": mae, "rmse": rmse, "baseline_mae": base})
    # return all folds as a small table
    return pd.DataFrame(rows)

# sanity check: rerun the GROUPED folds with the new loop
results_grouped_check = run_split("fold")
# show per-fold scores, rounded
print(results_grouped_check.round(1))
# mean of the 3 score columns
print(results_grouped_check.drop(columns="fold").mean().round(2))

In [ ]:
# run the SAME loop on the random folds
results_random = run_split("fold_random")
# per-fold scores, rounded
print(results_random.round(1))
# mean of each score column
print(results_random.drop(columns="fold").mean().round(2))
# SD across the 5 folds (pandas std, same as the report)
print(results_random.drop(columns="fold").std().round(2))
# SD of tonight's grouped rerun, for the side-by-side table
print(results_grouped_check.drop(columns="fold").std().round(2))

In [ ]:
# label each table with its split name
results_grouped_check["split"] = "grouped"
# same for the random results
results_random["split"] = "random"
# stack both tables into one
split_comparison = pd.concat([results_grouped_check, results_random])
# save to the working folder
split_comparison.to_csv("gi_split_comparison.csv", index=False)
# show what was saved
print(split_comparison.round(2))

In [1]:
import os  # load the module for listing files

for name in os.listdir("/kaggle/working"):  # go through every file in the working folder
    size_mb = os.path.getsize("/kaggle/working/" + name) / 1e6  # file size in megabytes
    print(name, round(size_mb, 1), "MB")  # print the name and size